In [1]:
import warnings
warnings.filterwarnings("ignore")
from sklearn.ensemble import RandomForestClassifier
import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification

In [2]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [3]:
# Need to choose patient_id from OUS_D1 in response_OUS
data = list(OUS_D1['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D2 with response_OUS
clinical_train = pd.merge(OUS_D1, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS', 'LRC', 'event_LRC'])]

In [4]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [5]:
# Check null values in D1
clinical_train.isnull().sum().sum()

0

In [6]:
""" Takes too long time 
# Check collinearity for OUS data 
df = clinical_train

# Create a correlation matrix
correlation_matrix = df.corr()

plt.figure(figsize=(15, 12))

# Plot a heatmap
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.show()
""" 

" Takes too long time \n# Check collinearity for OUS data \ndf = clinical_train\n\n# Create a correlation matrix\ncorrelation_matrix = df.corr()\n\nplt.figure(figsize=(15, 12))\n\n# Plot a heatmap\nsns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)\nplt.show()\n"

### COX assumption in Train data

In [7]:
df = pd.DataFrame(clinical_train)

# Fit Cox proportional hazards model
cph = CoxPHFitter(penalizer=0.00001)
cph.fit(df, duration_col='OS', event_col='event_OS')

# Perform the proportional hazards assumption test
results = proportional_hazard_test(cph, df)
print(results)

<lifelines.CoxPHFitter: fitted with 139 total observations, 82 right-censored observations>

<lifelines.StatisticalResult: proportional_hazard_test>
    time_transform = rank
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 139 total observations, 82 right-censored observations>
         test_name = proportional_hazard_test

---
                test_statistic      p  -log2(p)
MTV                       0.07   0.80      0.33
SUVpeak                   0.17   0.68      0.56
TLG                       0.38   0.54      0.90
age                       0.09   0.77      0.38
cavum_oris                0.00   0.99      0.02
charlson                  1.66   0.20      2.34
female                    2.67   0.10      3.29
histgrade_high            0.13   0.72      0.48
hpv_related              12.67 <0.005     11.39
hypopharynx               0.00   0.99      0.01
larynx                    0.00   0.98      0.03
oropharynx                0.00   0.98      0.02
pack_years                0.06   0.81      0.31
uicc8_III-IV              

In [8]:
# Access the summary table
summary_table = results.summary

# Filter columns based on p-values lower than 0.05
significant_columns = summary_table[summary_table['p'] < 0.05].index

# Display significant columns
print("Columns with p-values < 0.05:")
print(significant_columns)

Columns with p-values < 0.05:
Index(['hpv_related'], dtype='object')


In [9]:
df = pd.DataFrame(clinical_train)

# Fit Cox proportional hazards model
cph = CoxPHFitter(penalizer=0.1)
cph.fit(df, duration_col='OS', event_col='event_OS')

# Perform the proportional hazards assumption test
results = proportional_hazard_test(cph, df)
print(results)

<lifelines.CoxPHFitter: fitted with 139 total observations, 82 right-censored observations>

<lifelines.StatisticalResult: proportional_hazard_test>
    time_transform = rank
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 139 total observations, 82 right-censored observations>
         test_name = proportional_hazard_test

---
                test_statistic    p  -log2(p)
MTV                       0.59 0.44      1.17
SUVpeak                   0.01 0.91      0.14
TLG                       0.00 0.96      0.06
age                       0.43 0.51      0.97
cavum_oris                0.41 0.52      0.94
charlson                  0.87 0.35      1.51
female                    2.39 0.12      3.04
histgrade_high            0.11 0.74      0.44
hpv_related               6.46 0.01      6.51
hypopharynx               0.04 0.84      0.25
larynx                    0.97 0.33      1.62
oropharynx                0.85 0.36      1.49
pack_years                0.00 0.99      0.01
uicc8_III-IV              0.00 0.95      0.07


In [10]:
# Access the summary table
summary_table = results.summary

# Filter columns based on p-values lower than 0.05
significant_columns = summary_table[summary_table['p'] < 0.05].index

# Display significant columns
print("Columns with p-values < 0.05:")
print(significant_columns)

Columns with p-values < 0.05:
Index(['hpv_related'], dtype='object')


###### penalizer values essentially result in same result 

## Test dataset: MAASTRO 

In [11]:
(MAASTRO_D1['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [12]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [13]:
# need to choose patient_id from MAASTRO_D1 in response_MAASTRO
data = list(MAASTRO_D1['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 
response_MAASTRO

,patient_id,OS,OS_event,LRC,LRC_event,DFS,DFS_event
0,1,62.43,0.0,62.43,0.0,62.43,0.0
1,2,60.00,0.0,60.00,0.0,60.00,0.0
2,3,44.43,1.0,8.83,1.0,8.83,1.0
3,4,37.20,1.0,19.37,0.0,19.73,1.0
5,6,59.23,0.0,59.23,0.0,59.23,0.0
...,...,...,...,...,...,...,...
109,110,19.00,1.0,13.27,1.0,13.27,1.0
110,111,85.87,1.0,82.83,0.0,85.87,1.0
111,112,42.87,0.0,42.87,0.0,42.87,0.0
112,113,58.93,0.0,58.93,0.0,58.93,0.0


In [14]:
# Merge MAASTRO_D2 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D1, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'DFS_event', 'LRC', 'LRC_event'])]
clinical_test

,patient_id,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG,OS,OS_event
0,1,55,0,0,1,0,0,1,1,1,0,0,15.438583,22.841,263.611623,62.43,0.0
1,2,55,0,0,1,0,0,0,0,0,20,1,8.829353,5.660,36.980700,60.00,0.0
2,3,55,0,0,1,0,0,0,0,1,6,1,13.476123,7.791,74.636342,44.43,1.0
3,4,61,1,0,0,0,1,1,0,1,45,1,8.632732,7.908,46.791979,37.20,1.0
4,6,70,0,0,1,0,0,1,1,1,59,0,9.783954,15.237,107.637514,59.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,66,1,0,0,0,1,0,0,0,55,1,31.338410,6.110,144.490782,19.00,1.0
95,111,63,0,0,0,0,1,0,0,1,174,1,13.041604,7.182,69.214868,85.87,1.0
96,112,63,0,0,1,0,0,1,1,1,0,1,8.944517,16.483,102.594274,42.87,0.0
97,113,54,0,0,1,0,0,1,1,0,0,0,14.184236,9.981,103.229492,58.93,0.0


In [15]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [16]:
# Some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG,OS,OS_event


In [17]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS'])]

# y 
y = clinical_train.loc[:, ['OS', 'event_OS']]

In [18]:
# Set lower, upper time point and times for IBS calculation later 
lower, upper = np.percentile(y['OS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [19]:
# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

X_train:  (139, 14)
y_train:  (139,)


In [20]:
clinical_test.rename(columns = {'OS_event' : 'event_OS'}, inplace = True)

In [21]:
# Set X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'event_OS'])]

# Set y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['OS', 'event_OS']]

# Change y_MAASTRO into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_OS'], y_MAASTRO['OS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

(99, 16)

In [22]:
# VIF dataframe 
vif_data = pd.DataFrame() 
vif_data["feature"] = X.columns 
  
# calculating VIF for each feature 
vif_data["VIF"] = [variance_inflation_factor(X.values, i) 
                          for i in range(len(X.columns))] 

vif_data

,feature,VIF
0,age,1.136852
1,female,1.197250
2,cavum_oris,7.993011
3,oropharynx,60.199385
4,hypopharynx,11.118710
5,larynx,14.183474
6,histgrade_high,1.150479
7,hpv_related,4.309385
8,charlson,1.302350
9,pack_years,1.647441


# Standardization

In [23]:
original_X = X.copy()

In [24]:
# Standardize the data 
## Save the column and index 
X_columns = X.columns 
X_index = X.index

## Standardize the data but not the categorical columns
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']
### Separate the categorical and non-categorical columns
X_categorical = X[categorical_columns]
X_numeric = X.drop(categorical_columns, axis=1)
X_numeric_columns = X_numeric.columns
X_numeric_index = X_numeric.index

### Standardize non-categorical and then concat with the categorical
scaler = StandardScaler()  
X_numeric_std = scaler.fit_transform(X_numeric)
X_numeric_std = pd.DataFrame(X_numeric_std, columns=X_numeric_columns, index=X_numeric_index)
X_std = pd.concat([X_categorical, X_numeric_std], axis=1)

## Sort the order of the columns as it was in the clinical train 
X_std = X_std[original_X.columns]

In [31]:
# Standardize X_MAASTRO 
MAASTRO_new = X_MAASTRO.copy()
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns

MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = scaler.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
X_new_std = X_new_std[X_new.columns]
MAASTRO_new = MAASTRO_new[X_new.columns]
MAASTRO_new_std = MAASTRO_new_std[X_new.columns]

In [32]:
# Saving the data 
X_new = X 
X_new_std = X_std 
MAASTRO_new = X_MAASTRO 
MAASTRO_new_std = MAASTRO_new_std 

In [33]:
X_new

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG
0,54.238356,1,0,1,0,0,1,0.0,0,0.000000,0.0,14.473272,7.934,86.228420
1,54.539726,0,0,0,0,1,0,0.0,1,27.404795,0.0,5.044678,1.656,7.040100
2,59.019178,0,1,0,0,0,1,0.0,1,41.019178,1.0,7.839043,14.502,83.569669
3,70.726027,0,0,0,0,1,0,0.0,1,37.500000,0.0,2.880631,2.440,5.567091
4,67.865753,0,0,0,0,1,0,0.0,1,53.000000,0.0,5.402006,3.668,16.150550
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134,60.435616,0,0,1,0,0,1,1.0,0,0.000000,0.0,9.290139,3.650,26.280140
135,68.794521,0,0,1,0,0,1,1.0,0,0.000000,1.0,7.172883,18.967,101.754834
136,57.498630,0,0,1,0,0,1,1.0,1,39.498630,0.0,13.873187,6.370,66.273201
137,65.684932,0,0,1,0,0,1,1.0,1,71.527397,1.0,7.507419,12.443,71.832443


In [34]:
X_new_std

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG
0,-0.776210,1,0,1,0,0,1,0.0,0,-1.101176,0.0,0.646616,-0.296907,-0.179158
1,-0.737256,0,0,0,0,1,0,0.0,1,0.105775,0.0,-1.098689,-0.763433,-0.587333
2,-0.158251,0,1,0,0,0,1,0.0,1,0.705374,1.0,-0.581431,0.191169,-0.192863
3,1.354953,0,0,0,0,1,0,0.0,1,0.550384,0.0,-1.499271,-0.705173,-0.594926
4,0.985240,0,0,0,0,1,0,0.0,1,1.233028,0.0,-1.032545,-0.613919,-0.540373
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134,0.024835,0,0,1,0,0,1,1.0,0,-1.101176,0.0,-0.312822,-0.615257,-0.488161
135,1.105290,0,0,1,0,0,1,1.0,0,-1.101176,1.0,-0.704742,0.522969,-0.099128
136,-0.354794,0,0,1,0,0,1,1.0,1,0.638406,0.0,0.535536,-0.413130,-0.282017
137,0.703351,0,0,1,0,0,1,1.0,1,2.049005,1.0,-0.642817,0.038162,-0.253362


In [35]:
MAASTRO_new

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG
0,55,0,0,1,0,0,1,1,1,0,0,15.438583,22.841,263.611623
1,55,0,0,1,0,0,0,0,0,20,1,8.829353,5.660,36.980700
2,55,0,0,1,0,0,0,0,1,6,1,13.476123,7.791,74.636342
3,61,1,0,0,0,1,1,0,1,45,1,8.632732,7.908,46.791979
4,70,0,0,1,0,0,1,1,1,59,0,9.783954,15.237,107.637514
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,66,1,0,0,0,1,0,0,0,55,1,31.338410,6.110,144.490782
95,63,0,0,0,0,1,0,0,1,174,1,13.041604,7.182,69.214868
96,63,0,0,1,0,0,1,1,1,0,1,8.944517,16.483,102.594274
97,54,0,0,1,0,0,1,1,0,0,0,14.184236,9.981,103.229492


In [36]:
MAASTRO_new_std

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG
0,-0.677762,0,0,1,0,0,1,1,1,-1.101176,0,0.825303,0.810851,0.735160
1,-0.677762,0,0,1,0,0,0,0,0,-0.220344,1,-0.398117,-0.465891,-0.433005
2,-0.677762,0,0,1,0,0,0,0,1,-0.836927,1,0.462036,-0.307534,-0.238909
3,0.097786,1,0,0,0,1,1,0,1,0.880696,1,-0.434513,-0.298839,-0.382433
4,1.261108,0,0,1,0,0,1,1,1,1.497278,0,-0.221413,0.245788,-0.068806
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,0.744076,1,0,0,0,1,0,0,0,1.321112,1,3.768483,-0.432451,0.121154
95,0.356302,0,0,0,0,1,0,0,1,6.562062,1,0.381603,-0.352789,-0.266854
96,0.356302,0,0,1,0,0,1,1,1,-1.101176,1,-0.376799,0.338380,-0.094801
97,-0.807020,0,0,1,0,0,1,1,0,-1.101176,0,0.593113,-0.144792,-0.091527


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [37]:
# Setting the y format for skf below  
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-13 23:49:40,842] A new study created in memory with name: no-name-847ac761-860d-4d08-b266-a2ade24b8c99


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.7359307359307359
Fold 2 C-index: 0.7455357142857143
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.8523206751054853


[I 2024-04-13 23:49:43,079] A new study created in memory with name: no-name-075c332f-c652-49f2-81ec-080a12fad5af


Fold 5 C-index: 0.6525821596244131
[I 2024-04-13 23:49:43,067] Trial 0 finished with value: 0.7580581707147598 and parameters: {}. Best is trial 0 with value: 0.7580581707147598.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7580581707147598], datetime_start=datetime.datetime(2024, 4, 13, 23, 49, 41, 33018), datetime_complete=datetime.datetime(2024, 4, 13, 23, 49, 43, 66270), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7580581707147598


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.18224790157453516
Fold 2 IBS: 0.17936175872826987
Fold 3 IBS: 0.15783244652723072
Fold 4 IBS: 0.14729407963908242
Fold 5 IBS: 0.27375338992298043
[I 2024-04-13 23:49:44,363] Trial 0 finished with value: 0.18809791527841974 and parameters: {}. Best is trial 0 with value: 0.18809791527841974.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.18809791527841974], datetime_start=datetime.datetime(2024, 4, 13, 23, 49, 43, 227178), datetime_complete=datetime.datetime(2024, 4, 13, 23, 49, 44, 363151), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.18809791527841974


In [38]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [39]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.758
train_ibs:  0.188


#### Test

In [40]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [41]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.585
IBS score: 0.288


In [42]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [43]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis - Ridge

#### Train

In [44]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 23:49:44,913] A new study created in memory with name: no-name-f2e0514e-6f95-4bf9-b02f-e876ae54ad5a


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6212121212121212
Fold 2 C-index: 0.6383928571428571
Fold 3 C-index: 0.7720588235294118
Fold 4 C-index: 0.8037974683544303


[I 2024-04-13 23:49:45,434] A new study created in memory with name: no-name-63669f38-2793-4460-ac69-b1b99bdcc702


Fold 5 C-index: 0.5633802816901409
[I 2024-04-13 23:49:45,427] Trial 0 finished with value: 0.6797683103857922 and parameters: {}. Best is trial 0 with value: 0.6797683103857922.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6797683103857922], datetime_start=datetime.datetime(2024, 4, 13, 23, 49, 44, 966583), datetime_complete=datetime.datetime(2024, 4, 13, 23, 49, 45, 426821), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6797683103857922


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.21397651674697263
Fold 2 IBS: 0.22157791016872225
Fold 3 IBS: 0.20453594089893293
Fold 4 IBS: 0.2247380328252742
Fold 5 IBS: 0.21812431423859643
[I 2024-04-13 23:49:46,060] Trial 0 finished with value: 0.21659054297569966 and parameters: {}. Best is trial 0 with value: 0.21659054297569966.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.21659054297569966], datetime_start=datetime.datetime(2024, 4, 13, 23, 49, 45, 467562), datetime_complete=datetime.datetime(2024, 4, 13, 23, 49, 46, 60101), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.21659054297569966


In [45]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [46]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.68
train_ibs:  0.217


#### Test

In [47]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [48]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.534


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.221


In [49]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [50]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 23:49:46,737] A new study created in memory with name: no-name-0d2fe2c3-d7ba-4af1-a9b1-f1199db5d134


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.7359307359307359
Fold 2 C-index: 0.7455357142857143
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.8481012658227848


[I 2024-04-13 23:49:47,947] A new study created in memory with name: no-name-ad40fd59-2a90-47d1-b123-a146bfb2d38c


Fold 5 C-index: 0.6525821596244131
[I 2024-04-13 23:49:47,938] Trial 0 finished with value: 0.7581946810150825 and parameters: {}. Best is trial 0 with value: 0.7581946810150825.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7581946810150825], datetime_start=datetime.datetime(2024, 4, 13, 23, 49, 46, 850589), datetime_complete=datetime.datetime(2024, 4, 13, 23, 49, 47, 937941), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7581946810150825


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.1825807460821382
Fold 2 IBS: 0.17777166673446226
Fold 3 IBS: 0.15623537467337423
Fold 4 IBS: 0.1449349600534243
Fold 5 IBS: 0.27134181046825767
[I 2024-04-13 23:49:49,356] Trial 0 finished with value: 0.18657291160233133 and parameters: {}. Best is trial 0 with value: 0.18657291160233133.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.18657291160233133], datetime_start=datetime.datetime(2024, 4, 13, 23, 49, 47, 984104), datetime_complete=datetime.datetime(2024, 4, 13, 23, 49, 49, 356347), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.18657291160233133


In [51]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [52]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.758
train_ibs:  0.187


#### Test

In [53]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [54]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.59


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.284


In [55]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [56]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 23:49:50,249] A new study created in memory with name: no-name-71e23b88-a18b-4f95-ae2e-a1eaab6d5c62


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7359307359307359
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.8481012658227848
Fold 5 C-index: 0.6525821596244131
[I 2024-04-13 23:49:51,517] Trial 0 finished with value: 0.7573018238722253 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.7573018238722253.
Fold 1 C-index: 0.7359307359307359
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.8438818565400844
Fold 5 C-index: 0.6525821596244131
[I 2024-04-13 23:49:53,169] Trial 1 finished with value: 0.7564579420156853 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.7573018238722253.
Fold 1 C-index: 0.7359307359307359
Fold 2 C-index: 0.7366071428571429
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.8438818565400844
Fold 5 C-index: 0.6525821596244131
[I 2024-04-13 23:49:54,661] Trial 2 finished with value: 0.7555650848728281 and parameters: {'l1_ratio': 0.22692876841884668}.

Fold 1 C-index: 0.7359307359307359
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.8481012658227848
Fold 5 C-index: 0.6525821596244131
[I 2024-04-13 23:50:28,856] Trial 24 finished with value: 0.7573018238722253 and parameters: {'l1_ratio': 0.7805647035680947}. Best is trial 6 with value: 0.7581946810150825.
Fold 1 C-index: 0.7359307359307359
Fold 2 C-index: 0.7455357142857143
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.8481012658227848
Fold 5 C-index: 0.6525821596244131
[I 2024-04-13 23:50:30,138] Trial 25 finished with value: 0.7581946810150825 and parameters: {'l1_ratio': 0.9368557715647121}. Best is trial 6 with value: 0.7581946810150825.
Fold 1 C-index: 0.6277056277056277
Fold 2 C-index: 0.6294642857142857
Fold 3 C-index: 0.7598039215686274
Fold 4 C-index: 0.8227848101265823
Fold 5 C-index: 0.5915492957746479
[I 2024-04-13 23:50:30,514] Trial 26 finished with value: 0.6862615881779541 and parameters: {'l1_ratio': 0.01542375755129554

Fold 5 C-index: 0.6525821596244131
[I 2024-04-13 23:50:56,951] Trial 47 finished with value: 0.7581946810150825 and parameters: {'l1_ratio': 0.9991045282580079}. Best is trial 6 with value: 0.7581946810150825.
Fold 1 C-index: 0.7359307359307359
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.8481012658227848
Fold 5 C-index: 0.6525821596244131
[I 2024-04-13 23:50:59,058] Trial 48 finished with value: 0.7573018238722253 and parameters: {'l1_ratio': 0.8619031648167028}. Best is trial 6 with value: 0.7581946810150825.
Fold 1 C-index: 0.7359307359307359
Fold 2 C-index: 0.7455357142857143
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.8481012658227848
Fold 5 C-index: 0.6525821596244131
[I 2024-04-13 23:51:00,888] Trial 49 finished with value: 0.7581946810150825 and parameters: {'l1_ratio': 0.920787420171088}. Best is trial 6 with value: 0.7581946810150825.
Fold 1 C-index: 0.7359307359307359
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.808

...

Fold 1 IBS: 0.1826930793406474
Fold 2 IBS: 0.17814098168529346
Fold 3 IBS: 0.15607788318957047
Fold 4 IBS: 0.14507981109510232
Fold 5 IBS: 0.27092973904575407
[I 2024-04-13 23:52:30,073] Trial 0 finished with value: 0.18658429887127354 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.18658429887127354.
Fold 1 IBS: 0.18287199758336523
Fold 2 IBS: 0.17882425604642588
Fold 3 IBS: 0.1559432644144467
Fold 4 IBS: 0.14534435289145511
Fold 5 IBS: 0.2707544925394689
[I 2024-04-13 23:52:31,707] Trial 1 finished with value: 0.18674767269503237 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.18658429887127354.
Fold 1 IBS: 0.1829089514469264
Fold 2 IBS: 0.17896411692047573
Fold 3 IBS: 0.15600761798837737
Fold 4 IBS: 0.1453400489680181
Fold 5 IBS: 0.2705781041195937
[I 2024-04-13 23:52:33,463] Trial 2 finished with value: 0.18675976788867826 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 0 with value: 0.186584298871273

Fold 1 IBS: 0.182597095351905
Fold 2 IBS: 0.17780878281458193
Fold 3 IBS: 0.15621078519759202
Fold 4 IBS: 0.1450040193628384
Fold 5 IBS: 0.27127120030386354
[I 2024-04-13 23:53:15,305] Trial 25 finished with value: 0.18657837660615617 and parameters: {'l1_ratio': 0.9058787695685316}. Best is trial 15 with value: 0.18653078342548116.
Fold 1 IBS: 0.18258218281612165
Fold 2 IBS: 0.17777251677605124
Fold 3 IBS: 0.15623066423870974
Fold 4 IBS: 0.1449310825047383
Fold 5 IBS: 0.27133378912614853
[I 2024-04-13 23:53:16,817] Trial 26 finished with value: 0.18657004709235386 and parameters: {'l1_ratio': 0.9973554689957201}. Best is trial 15 with value: 0.18653078342548116.
Fold 1 IBS: 0.1826243644080164
Fold 2 IBS: 0.17798356580742047
Fold 3 IBS: 0.15614784526094985
Fold 4 IBS: 0.1450431116078512
Fold 5 IBS: 0.27108642287726653
[I 2024-04-13 23:53:18,842] Trial 27 finished with value: 0.1865770619923009 and parameters: {'l1_ratio': 0.7927751225334085}. Best is trial 15 with value: 0.186530783425

Fold 5 IBS: 0.2712063814448089
[I 2024-04-13 23:54:09,302] Trial 49 finished with value: 0.18655607739951546 and parameters: {'l1_ratio': 0.9533747955148694}. Best is trial 41 with value: 0.18652670686255707.
Fold 1 IBS: 0.182694156414943
Fold 2 IBS: 0.1781668981429178
Fold 3 IBS: 0.15606469614602242
Fold 4 IBS: 0.1451557949146759
Fold 5 IBS: 0.27088556667520813
[I 2024-04-13 23:54:12,168] Trial 50 finished with value: 0.18659342245875343 and parameters: {'l1_ratio': 0.6355126009518985}. Best is trial 41 with value: 0.18652670686255707.
Fold 1 IBS: 0.18260945314808652
Fold 2 IBS: 0.17781788550647537
Fold 3 IBS: 0.15616783777756874
Fold 4 IBS: 0.14497161979587717
Fold 5 IBS: 0.2711896026040896
[I 2024-04-13 23:54:14,872] Trial 51 finished with value: 0.18655127976641944 and parameters: {'l1_ratio': 0.8830080032072889}. Best is trial 41 with value: 0.18652670686255707.
Fold 1 IBS: 0.18265223715921625
Fold 2 IBS: 0.17797428213237454
Fold 3 IBS: 0.15618477542954717
Fold 4 IBS: 0.1450764919

Fold 1 IBS: 0.18261691621706228
Fold 2 IBS: 0.17782098343932493
Fold 3 IBS: 0.15614159814021916
Fold 4 IBS: 0.14495196889022768
Fold 5 IBS: 0.2711424711303331
[I 2024-04-13 23:55:20,543] Trial 74 finished with value: 0.1865347875634334 and parameters: {'l1_ratio': 0.8693131368936011}. Best is trial 41 with value: 0.18652670686255707.
Fold 1 IBS: 0.18265769389109618
Fold 2 IBS: 0.17800469504415845
Fold 3 IBS: 0.1561742081163794
Fold 4 IBS: 0.14495671229029594
Fold 5 IBS: 0.27111397070819115
[I 2024-04-13 23:55:22,407] Trial 75 finished with value: 0.1865814560100242 and parameters: {'l1_ratio': 0.7410231761145568}. Best is trial 41 with value: 0.18652670686255707.
Fold 1 IBS: 0.18259435732656015
Fold 2 IBS: 0.1778066685259249
Fold 3 IBS: 0.15622065159803722
Fold 4 IBS: 0.14501125481224078
Fold 5 IBS: 0.27128993626355785
[I 2024-04-13 23:55:24,142] Trial 76 finished with value: 0.18658457370526418 and parameters: {'l1_ratio': 0.9112874259530965}. Best is trial 41 with value: 0.1865267068

Fold 1 IBS: 0.1826438584305466
Fold 2 IBS: 0.17796876109479737
Fold 3 IBS: 0.15621198957389612
Fold 4 IBS: 0.1450966199403859
Fold 5 IBS: 0.2712203585949212
[I 2024-04-13 23:56:21,382] Trial 99 finished with value: 0.18662831752690945 and parameters: {'l1_ratio': 0.8277760149481713}. Best is trial 77 with value: 0.18652409062144285.


* Best trial for IBS: 
 FrozenTrial(number=77, state=TrialState.COMPLETE, values=[0.18652409062144285], datetime_start=datetime.datetime(2024, 4, 13, 23, 55, 24, 148756), datetime_complete=datetime.datetime(2024, 4, 13, 23, 55, 26, 44610), params={'l1_ratio': 0.8606102150962986}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'l1_ratio': FloatDistribution(high=1.0, log=False, low=0.0001, step=None)}, trial_id=77, value=None)


* Best Score for IBS: 
 0.18652409062144285


In [57]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [58]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.758
train_ibs:  0.187


#### Test

In [59]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [60]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.980766121964777)

test_cindex : 0.59


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.8606102150962986)

test_ibs:  0.284


In [61]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [62]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-13 23:56:23,640] A new study created in memory with name: no-name-9cdbe7e8-920a-44ee-8e77-ea6ac4865b2c


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7532467532467533
Fold 2 C-index: 0.6339285714285714
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.8544303797468354
Fold 5 C-index: 0.6408450704225352
[I 2024-04-13 23:56:43,901] Trial 0 finished with value: 0.7372744686944293 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.7372744686944293.
Fold 1 C-index: 0.7012987012987013
Fold 2 C-index: 0.6875
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.8607594936708861
Fold 5 C-index: 0.647887323943662
[I 2024-04-13 23:56:55,935] Trial 1 finished with value: 0.74027341750814 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, 'max_features': '

Fold 1 C-index: 0.70995670995671
Fold 2 C-index: 0.8392857142857143
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.8755274261603375
Fold 5 C-index: 0.7230046948356808
[I 2024-04-13 23:59:27,452] Trial 16 finished with value: 0.7913196149300414 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 15, 'min_samples_leaf': 16, 'max_depth': 2, 'n_estimators': 62, 'oob_score': True, 'max_samples': 0.9622193970781673, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.07748782870655765, 'warm_start': True}. Best is trial 14 with value: 0.8012393138587386.
Fold 1 C-index: 0.6796536796536796
Fold 2 C-index: 0.8571428571428571
Fold 3 C-index: 0.8137254901960784
Fold 4 C-index: 0.8839662447257384
Fold 5 C-index: 0.7417840375586855
[I 2024-04-13 23:59:32,134] Trial 17 finished with value: 0.7952544618554078 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 16, 'min_samples_leaf': 15, 'max_depth': 7, 'n_estimators': 127, 'oob_score': True, 'max_samples': 0.8172323984404596, 

Fold 1 C-index: 0.6753246753246753
Fold 2 C-index: 0.8571428571428571
Fold 3 C-index: 0.8137254901960784
Fold 4 C-index: 0.9071729957805907
Fold 5 C-index: 0.7981220657276995
[I 2024-04-14 00:01:44,913] Trial 31 finished with value: 0.8102976168343801 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 7, 'min_samples_leaf': 7, 'max_depth': 12, 'n_estimators': 317, 'oob_score': True, 'max_samples': 0.8965443067001324, 'max_features': None, 'min_weight_fraction_leaf': 0.04869831874272522, 'warm_start': True}. Best is trial 30 with value: 0.8141045941707695.
Fold 1 C-index: 0.6883116883116883
Fold 2 C-index: 0.8571428571428571
Fold 3 C-index: 0.7990196078431373
Fold 4 C-index: 0.9029535864978903
Fold 5 C-index: 0.7746478873239436
[I 2024-04-14 00:01:54,804] Trial 32 finished with value: 0.8044151254239035 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 6, 'min_samples_leaf': 7, 'max_depth': 13, 'n_estimators': 317, 'oob_score': True, 'max_samples': 0.7822231835771817,

...

Fold 1 IBS: 0.21398563901141657
Fold 2 IBS: 0.22134596740550985
Fold 3 IBS: 0.2048025990579434
Fold 4 IBS: 0.22461415055237854
Fold 5 IBS: 0.21851803782809134
[I 2024-04-14 00:36:18,709] Trial 32 finished with value: 0.21665327877106794 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 10, 'min_samples_leaf': 3, 'max_depth': 18, 'n_estimators': 451, 'oob_score': False, 'max_samples': 0.7396444271446992, 'max_features': None, 'min_weight_fraction_leaf': 0.4501495537267681}. Best is trial 23 with value: 0.1829201586199189.
Fold 1 IBS: 0.19337121417130673
Fold 2 IBS: 0.19511882700454072
Fold 3 IBS: 0.18642967451701625
Fold 4 IBS: 0.1895345717679471
Fold 5 IBS: 0.21856358775020004
[I 2024-04-14 00:36:39,382] Trial 33 finished with value: 0.19660357504220216 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 12, 'min_samples_leaf': 1, 'max_depth': 19, 'n_estimators': 493, 'oob_score': False, 'max_samples': 0.9933327757837453, 'max_features': None, 'min_weight_fraction_leaf

Fold 1 IBS: 0.21398162469590376
Fold 2 IBS: 0.22136781149843104
Fold 3 IBS: 0.20480753036595514
Fold 4 IBS: 0.22461586717115936
Fold 5 IBS: 0.21858421480458282
[I 2024-04-14 00:41:05,934] Trial 48 finished with value: 0.21667140970720644 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 12, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 459, 'oob_score': False, 'max_samples': 0.8399357435042919, 'max_features': None, 'min_weight_fraction_leaf': 0.43949783817145754}. Best is trial 23 with value: 0.1829201586199189.
Fold 1 IBS: 0.18607291644289184
Fold 2 IBS: 0.1997166990883884
Fold 3 IBS: 0.17871196581310303
Fold 4 IBS: 0.1866755623669813
Fold 5 IBS: 0.21401906861191788
[I 2024-04-14 00:41:24,068] Trial 49 finished with value: 0.19303924246465648 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 6, 'min_samples_leaf': 20, 'max_depth': 6, 'n_estimators': 480, 'oob_score': False, 'max_samples': 0.9976678104279734, 'max_features': 'auto', 'min_weight_fraction_lea

Fold 1 IBS: 0.17214163753608527
Fold 2 IBS: 0.19020363580055616
Fold 3 IBS: 0.16550959046326688
Fold 4 IBS: 0.16526794959392554
Fold 5 IBS: 0.22833087784482511
[I 2024-04-14 00:44:45,267] Trial 64 finished with value: 0.1842907382477318 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 14, 'min_samples_leaf': 7, 'max_depth': 18, 'n_estimators': 359, 'oob_score': False, 'max_samples': 0.8662053800926933, 'max_features': None, 'min_weight_fraction_leaf': 0.39542258978805284}. Best is trial 23 with value: 0.1829201586199189.
Fold 1 IBS: 0.2139792007492792
Fold 2 IBS: 0.22131668306329724
Fold 3 IBS: 0.20481655835559517
Fold 4 IBS: 0.22465389902940344
Fold 5 IBS: 0.2186107338002615
[I 2024-04-14 00:44:56,246] Trial 65 finished with value: 0.21667541499956733 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 12, 'min_samples_leaf': 8, 'max_depth': 20, 'n_estimators': 370, 'oob_score': False, 'max_samples': 0.8126605473414451, 'max_features': None, 'min_weight_fraction_leaf'

Fold 1 IBS: 0.21398666839515476
Fold 2 IBS: 0.22133561292920703
Fold 3 IBS: 0.20476084552868884
Fold 4 IBS: 0.22460858228328426
Fold 5 IBS: 0.21850495327987215
[I 2024-04-14 00:48:01,426] Trial 80 finished with value: 0.2166393324832414 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 14, 'min_samples_leaf': 6, 'max_depth': 16, 'n_estimators': 442, 'oob_score': True, 'max_samples': 0.7543724309332009, 'max_features': None, 'min_weight_fraction_leaf': 0.43164085935118673}. Best is trial 23 with value: 0.1829201586199189.
Fold 1 IBS: 0.1718240887458899
Fold 2 IBS: 0.191437443203446
Fold 3 IBS: 0.16491345446124533
Fold 4 IBS: 0.16398408770898562
Fold 5 IBS: 0.22656733476064983
[I 2024-04-14 00:48:14,310] Trial 81 finished with value: 0.18374528177604332 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 14, 'min_samples_leaf': 8, 'max_depth': 18, 'n_estimators': 355, 'oob_score': False, 'max_samples': 0.8723363913158229, 'max_features': None, 'min_weight_fraction_leaf': 

Fold 1 IBS: 0.18206223269633873
Fold 2 IBS: 0.18820585180072097
Fold 3 IBS: 0.17369714014143833
Fold 4 IBS: 0.1751220212365198
Fold 5 IBS: 0.22437209861320465
[I 2024-04-14 00:51:32,701] Trial 96 finished with value: 0.1886918688976445 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 19, 'min_samples_leaf': 1, 'max_depth': 8, 'n_estimators': 448, 'oob_score': False, 'max_samples': 0.8430407084230132, 'max_features': None, 'min_weight_fraction_leaf': 0.41419337591128474}. Best is trial 23 with value: 0.1829201586199189.
Fold 1 IBS: 0.17261810934600746
Fold 2 IBS: 0.19291443628076307
Fold 3 IBS: 0.16469348295311864
Fold 4 IBS: 0.16293280131529528
Fold 5 IBS: 0.2265701418039605
[I 2024-04-14 00:51:49,198] Trial 97 finished with value: 0.183945794339829 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 15, 'min_samples_leaf': 3, 'max_depth': 9, 'n_estimators': 499, 'oob_score': False, 'max_samples': 0.8784577990568653, 'max_features': None, 'min_weight_fraction_leaf': 0

In [63]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [64]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.826
train_ibs:  0.183


#### Test

In [65]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

In [66]:
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])
 
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=19, max_features='log2', max_leaf_nodes=15,
                     max_samples=0.9995515317027949, min_samples_split=18,
                     min_weight_fraction_leaf=0.032175807966512454,
                     n_estimators=115, oob_score=True, random_state=123,
                     warm_start=True)

test_cindex:  0.653


RandomSurvivalForest(max_depth=20, max_features=None, max_leaf_nodes=13,
                     max_samples=0.9248759152889626, min_samples_split=2,
                     min_weight_fraction_leaf=0.41894325026727597,
                     n_estimators=396, random_state=123)

test_ibs:  0.206


In [67]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [68]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

In [69]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 00:52:17,967] A new study created in memory with name: no-name-9e6e472a-40ec-4e0c-a4ad-55f59e8e50d4


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7359307359307359
Fold 2 C-index: 0.8080357142857143
Fold 3 C-index: 0.8480392156862745
Fold 4 C-index: 0.8312236286919831
Fold 5 C-index: 0.6948356807511737
[I 2024-04-14 00:52:21,038] Trial 0 finished with value: 0.7836129950691764 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.7836129950691764.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 00:52:28,101] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531

Fold 3 C-index: 0.8284313725490197
Fold 4 C-index: 0.8122362869198312
Fold 5 C-index: 0.6713615023474179
[I 2024-04-14 00:53:46,034] Trial 15 finished with value: 0.7684393821468035 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 9, 'min_samples_leaf': 13, 'max_depth': 4, 'n_estimators': 258, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.6880212408103346, 'min_weight_fraction_leaf': 0.07884420972256234}. Best is trial 12 with value: 0.7914452579440178.
Fold 1 C-index: 0.7662337662337663
Fold 2 C-index: 0.8013392857142857
Fold 3 C-index: 0.8333333333333334
Fold 4 C-index: 0.8248945147679325
Fold 5 C-index: 0.6267605633802817
[I 2024-04-14 00:53:49,690] Trial 16 finished with value: 0.77051229268592 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 499, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.8750346354626457, 'min_weight_fraction_l

Fold 1 C-index: 0.7489177489177489
Fold 2 C-index: 0.7924107142857143
Fold 3 C-index: 0.8186274509803921
Fold 4 C-index: 0.8185654008438819
Fold 5 C-index: 0.6572769953051644
[I 2024-04-14 00:54:47,856] Trial 30 finished with value: 0.7671596620665804 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 11, 'min_samples_leaf': 12, 'max_depth': 5, 'n_estimators': 459, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.8382504072008882, 'min_weight_fraction_leaf': 0.23846812660748434}. Best is trial 12 with value: 0.7914452579440178.
Fold 1 C-index: 0.7229437229437229
Fold 2 C-index: 0.8169642857142857
Fold 3 C-index: 0.8529411764705882
Fold 4 C-index: 0.8248945147679325
Fold 5 C-index: 0.6384976525821596
[I 2024-04-14 00:54:55,600] Trial 31 finished with value: 0.7712482704957379 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 5, 'min_samples_leaf': 16, 'max_depth': 13, 'n_estimators': 500, 'oob_score': True, 'warm_start': True, 'max_features'

...

Fold 1 IBS: 0.16792251267623906
Fold 2 IBS: 0.2174484716230587
Fold 3 IBS: 0.16273781989489802
Fold 4 IBS: 0.1558940391227505
Fold 5 IBS: 0.23222422254692077
[I 2024-04-14 01:06:24,719] Trial 30 finished with value: 0.18724541317277338 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 6, 'min_samples_leaf': 2, 'max_depth': 12, 'n_estimators': 455, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.7293028765390641, 'min_weight_fraction_leaf': 0.08037045697723012}. Best is trial 23 with value: 0.1855727116611247.
Fold 1 IBS: 0.16646920043412164
Fold 2 IBS: 0.22144555868438975
Fold 3 IBS: 0.1612585234507463
Fold 4 IBS: 0.15276461377317607
Fold 5 IBS: 0.23760424924495455
[I 2024-04-14 01:06:30,889] Trial 31 finished with value: 0.18790842911747765 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 15, 'min_samples_leaf': 1, 'max_depth': 5, 'n_estimators': 209, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.83

Fold 1 IBS: 0.16512803721910052
Fold 2 IBS: 0.21893230349393553
Fold 3 IBS: 0.1593836385256823
Fold 4 IBS: 0.15058205502042205
Fold 5 IBS: 0.23336241815587846
[I 2024-04-14 01:08:09,628] Trial 45 finished with value: 0.18547769048300378 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 6, 'min_samples_leaf': 3, 'max_depth': 12, 'n_estimators': 276, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.6931500413934508, 'min_weight_fraction_leaf': 0.021457257714753555}. Best is trial 32 with value: 0.18528124104479504.
Fold 1 IBS: 0.1659546836323674
Fold 2 IBS: 0.2153197227930472
Fold 3 IBS: 0.16360867863772716
Fold 4 IBS: 0.15557836846786757
Fold 5 IBS: 0.233640633124344
[I 2024-04-14 01:08:17,773] Trial 46 finished with value: 0.18682041733107066 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 13, 'min_samples_leaf': 6, 'max_depth': 13, 'n_estimators': 283, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0

Fold 1 IBS: 0.17342492293804432
Fold 2 IBS: 0.2110732537830341
Fold 3 IBS: 0.16831819527481992
Fold 4 IBS: 0.16873369129336893
Fold 5 IBS: 0.22655614890084055
[I 2024-04-14 01:10:37,507] Trial 60 finished with value: 0.18962124243802156 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 4, 'min_samples_leaf': 5, 'max_depth': 14, 'n_estimators': 439, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.5735294094022945, 'min_weight_fraction_leaf': 0.10077092974890593}. Best is trial 53 with value: 0.18257594357755064.
Fold 1 IBS: 0.16275315104094723
Fold 2 IBS: 0.22028966274589679
Fold 3 IBS: 0.16253743592400494
Fold 4 IBS: 0.15352895984062664
Fold 5 IBS: 0.23021227496706434
[I 2024-04-14 01:10:48,256] Trial 61 finished with value: 0.18586429690370798 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 4, 'min_samples_leaf': 3, 'max_depth': 11, 'n_estimators': 412, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples':

Fold 1 IBS: 0.16278536069617075
Fold 2 IBS: 0.22061911570320727
Fold 3 IBS: 0.15948095174532076
Fold 4 IBS: 0.1500453215492157
Fold 5 IBS: 0.23108140998200039
[I 2024-04-14 01:13:29,443] Trial 75 finished with value: 0.18480243193518295 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 5, 'min_samples_leaf': 2, 'max_depth': 13, 'n_estimators': 463, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.6487943567829829, 'min_weight_fraction_leaf': 0.011079149496446542}. Best is trial 53 with value: 0.18257594357755064.
Fold 1 IBS: 0.1649931201311961
Fold 2 IBS: 0.2191098546301487
Fold 3 IBS: 0.1604180424655896
Fold 4 IBS: 0.15133070246635846
Fold 5 IBS: 0.23270813963892614
[I 2024-04-14 01:13:46,589] Trial 76 finished with value: 0.18571197186644378 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 8, 'min_samples_leaf': 2, 'max_depth': 13, 'n_estimators': 496, 'oob_score': True, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0

Fold 1 IBS: 0.18162653052097805
Fold 2 IBS: 0.20612651784753333
Fold 3 IBS: 0.1771784733434652
Fold 4 IBS: 0.18941755917999278
Fold 5 IBS: 0.21614311489575638
[I 2024-04-14 01:16:18,367] Trial 90 finished with value: 0.19409843915754516 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 6, 'min_samples_leaf': 1, 'max_depth': 20, 'n_estimators': 362, 'oob_score': True, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0.7635642567316503, 'min_weight_fraction_leaf': 0.280081595660318}. Best is trial 53 with value: 0.18257594357755064.
Fold 1 IBS: 0.16186089564069253
Fold 2 IBS: 0.21899259491528042
Fold 3 IBS: 0.15921485444738478
Fold 4 IBS: 0.148193042465612
Fold 5 IBS: 0.23190417972474658
[I 2024-04-14 01:16:29,136] Trial 91 finished with value: 0.18403311343874326 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 5, 'min_samples_leaf': 1, 'max_depth': 19, 'n_estimators': 323, 'oob_score': True, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0.789

In [70]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [71]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.794
train_ibs:  0.183


#### Test

In [72]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [73]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=12, max_features=None, max_leaf_nodes=6,
                   max_samples=0.7030590704740037, min_samples_leaf=6,
                   min_samples_split=13,
                   min_weight_fraction_leaf=0.020016091327628317,
                   n_estimators=273, random_state=123, warm_start=True)

C-index score: 0.618


ExtraSurvivalTrees(max_depth=13, max_features='log2', max_leaf_nodes=6,
                   max_samples=0.7048480399273186, min_samples_leaf=1,
                   min_samples_split=12,
                   min_weight_fraction_leaf=0.0010291888027469595,
                   n_estimators=485, random_state=123, warm_start=True)

IBS: 0.215


In [74]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [75]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-14 01:17:57,990] A new study created in memory with name: no-name-d6190f60-8ddb-4976-aa0f-21f1c3ad88ee


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 01:18:50,827] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 01:19:17,961] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 01:33:06,654] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 9 with value: 0.7334874350516556.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 01:34:50,374] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'square

Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 01:52:00,591] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.7565257046780437, 'learning_rate': 0.01188344684416992, 'dropout_rate': 0.3433523077110169, 'n_estimators': 318, 'criterion': 'squared_error', 'ccp_alpha': 2.063559212730723, 'min_weight_fraction_leaf': 0.25401273903421573, 'max_features': 'auto', 'min_impurity_decrease': 5.773435946665558e-07, 'validation_fraction': 0.8062268184400477, 'min_samples_split': 16, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 3}. Best is trial 9 with value: 0.7334874350516556.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.6004464285714286
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.6103286384976526
[I 2024-04-14 01:53:56,845] Trial 26 finished with value: 0.5421550134138162 and parameters: {'subsample': 0.8922404482621683, 'learning_rate': 0.011585260292674536, 'dropout_rate': 0.2527000999648632, 'n_estimators': 446,

...

Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609566
[I 2024-04-14 05:15:13,008] Trial 55 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.9027683411994925, 'learning_rate': 0.023646082998228058, 'dropout_rate': 0.13472529918097312, 'n_estimators': 381, 'criterion': 'squared_error', 'ccp_alpha': 1.3381569877935875, 'min_weight_fraction_leaf': 0.36938177041618503, 'max_features': 'auto', 'min_impurity_decrease': 1.1016843774656315e-07, 'validation_fraction': 0.898549324711475, 'min_samples_split': 3, 'max_leaf_nodes': 12, 'min_samples_leaf': 12, 'max_depth': 3}. Best is trial 53 with value: 0.2150432178150902.
Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609582
[I 2024-04-14 05:17:14,854] Trial 56 finished with value: 0.21659054862241592 and parameters: {'subsample': 0.8324789538052517, 'learning_rate': 0.0985092209

Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609563
[I 2024-04-14 05:42:04,215] Trial 66 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.9104313304151431, 'learning_rate': 0.014815622348446418, 'dropout_rate': 0.1415896279444537, 'n_estimators': 149, 'criterion': 'squared_error', 'ccp_alpha': 0.7060910424624821, 'min_weight_fraction_leaf': 0.2796836959018326, 'max_features': 'auto', 'min_impurity_decrease': 5.127880425748818e-06, 'validation_fraction': 0.916501041728546, 'min_samples_split': 19, 'max_leaf_nodes': 16, 'min_samples_leaf': 18, 'max_depth': 5}. Best is trial 53 with value: 0.2150432178150902.
Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609566
[I 2024-04-14 05:44:27,624] Trial 67 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.802369464636257, 'learning_rate': 0.0041755238500

Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609566
[I 2024-04-14 06:13:24,838] Trial 77 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.8921532985730489, 'learning_rate': 0.021931279910218017, 'dropout_rate': 0.19284931428290142, 'n_estimators': 490, 'criterion': 'squared_error', 'ccp_alpha': 0.5905545096948588, 'min_weight_fraction_leaf': 0.07057220045251793, 'max_features': 'log2', 'min_impurity_decrease': 1.637559261311236e-05, 'validation_fraction': 0.8150762480319586, 'min_samples_split': 5, 'max_leaf_nodes': 13, 'min_samples_leaf': 8, 'max_depth': 5}. Best is trial 53 with value: 0.2150432178150902.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609563
[I 2024-04-14 06:15:59,551] Trial 78 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.7443996478445325, 'learning_rate': 0.086932868663

Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609566
[I 2024-04-14 06:42:22,588] Trial 88 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.9447987915285996, 'learning_rate': 0.014159070275535094, 'dropout_rate': 0.18102387516231752, 'n_estimators': 470, 'criterion': 'squared_error', 'ccp_alpha': 0.5935000311483954, 'min_weight_fraction_leaf': 0.2151508441811604, 'max_features': 'auto', 'min_impurity_decrease': 3.72786238165322e-07, 'validation_fraction': 0.898328514849679, 'min_samples_split': 18, 'max_leaf_nodes': 18, 'min_samples_leaf': 12, 'max_depth': 2}. Best is trial 53 with value: 0.2150432178150902.
Fold 1 IBS: 0.21292314886383587
Fold 2 IBS: 0.2213935582997429
Fold 3 IBS: 0.20349492943164285
Fold 4 IBS: 0.2234443788901035
Fold 5 IBS: 0.2178743294294726
[I 2024-04-14 06:44:29,391] Trial 89 finished with value: 0.21582606898295956 and parameters: {'subsample': 0.9777744983689288, 'learning_rate': 0.010383571834622

Fold 3 IBS: 0.20237857642828805
Fold 4 IBS: 0.22105646622231043
Fold 5 IBS: 0.21749535106188492
[I 2024-04-14 06:59:30,909] Trial 99 finished with value: 0.21481876339222578 and parameters: {'subsample': 0.8569271041566342, 'learning_rate': 0.01377474153403631, 'dropout_rate': 0.2044520049398774, 'n_estimators': 378, 'criterion': 'squared_error', 'ccp_alpha': 0.004422201299274769, 'min_weight_fraction_leaf': 0.13940114558509004, 'max_features': 'auto', 'min_impurity_decrease': 0.00015837847703956763, 'validation_fraction': 0.9580200141092102, 'min_samples_split': 13, 'max_leaf_nodes': 17, 'min_samples_leaf': 13, 'max_depth': 2}. Best is trial 99 with value: 0.21481876339222578.


* Best trial for IBS: 
 FrozenTrial(number=99, state=TrialState.COMPLETE, values=[0.21481876339222578], datetime_start=datetime.datetime(2024, 4, 14, 6, 58, 8, 973367), datetime_complete=datetime.datetime(2024, 4, 14, 6, 59, 30, 908215), params={'subsample': 0.8569271041566342, 'learning_rate': 0.0137747415340

In [76]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [77]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.745
train_ibs:  0.215


#### Test

In [78]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [79]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.03063070291051248,
                                 criterion='squared_error',
                                 dropout_rate=0.2576884847115747,
                                 learning_rate=0.007359366951045268,
                                 max_depth=16, max_features=1,
                                 max_leaf_nodes=16,
                                 min_impurity_decrease=1.3903906488794697e-07,
                                 min_samples_leaf=14, min_samples_split=20,
                                 min_weight_fraction_leaf=0.38180626899357545,
                                 n_estimators=96, random_state=123,
                                 subsample=0.8938290428827321,
                                 validation_fraction=0.9577535215137098)

C-index score: 0.603


GradientBoostingSurvivalAnalysis(ccp_alpha=0.004422201299274769,
                                 criterion='squared_error',
                                 dropout_rate=0.2044520049398774,
                                 learning_rate=0.01377474153403631, max_depth=2,
                                 max_features='auto', max_leaf_nodes=17,
                                 min_impurity_decrease=0.00015837847703956763,
                                 min_samples_leaf=13, min_samples_split=13,
                                 min_weight_fraction_leaf=0.13940114558509004,
                                 n_estimators=378, random_state=123,
                                 subsample=0.8569271041566342,
                                 validation_fraction=0.9580200141092102)

IBS: 0.22


In [80]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [81]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

In [82]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 06:59:48,298] A new study created in memory with name: no-name-51961c76-435c-4880-a528-143cf2823f41


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7402597402597403
Fold 2 C-index: 0.6339285714285714
Fold 3 C-index: 0.7549019607843137
Fold 4 C-index: 0.8270042194092827
Fold 5 C-index: 0.5938967136150235
[I 2024-04-14 06:59:50,224] Trial 0 finished with value: 0.7099982410993864 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.7099982410993864.
Fold 1 C-index: 0.7402597402597403
Fold 2 C-index: 0.625
Fold 3 C-index: 0.7450980392156863
Fold 4 C-index: 0.7848101265822784
Fold 5 C-index: 0.5985915492957746
[I 2024-04-14 07:00:02,535] Trial 1 finished with value: 0.6987518910706959 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.7099982410993864.
Fold 1 C-index: 0.7445887445887446
Fold 2 C-index: 0.6294642857142857
Fold 3 C-index: 0.7794117647058824
Fold 4 C-index: 0.

Fold 1 C-index: 0.7186147186147186
Fold 2 C-index: 0.6651785714285714
Fold 3 C-index: 0.7990196078431373
Fold 4 C-index: 0.8312236286919831
Fold 5 C-index: 0.6103286384976526
[I 2024-04-14 07:01:46,968] Trial 19 finished with value: 0.7248730330152127 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.9824925186694111, 'n_estimators': 400, 'learning_rate': 0.08685281684248895}. Best is trial 15 with value: 0.7262366026411862.
Fold 1 C-index: 0.7402597402597403
Fold 2 C-index: 0.6383928571428571
Fold 3 C-index: 0.7549019607843137
Fold 4 C-index: 0.8270042194092827
Fold 5 C-index: 0.5985915492957746
[I 2024-04-14 07:01:50,582] Trial 20 finished with value: 0.7118300653783937 and parameters: {'subsample': 0.6343904688286152, 'dropout_rate': 0.7356300508281354, 'n_estimators': 258, 'learning_rate': 0.06551866052750378}. Best is trial 15 with value: 0.7262366026411862.
Fold 1 C-index: 0.7402597402597403
Fold 2 C-index: 0.6294642857142857
Fold 3 C-index: 0.7843137254901961


Fold 1 C-index: 0.7445887445887446
Fold 2 C-index: 0.6473214285714286
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.8270042194092827
Fold 5 C-index: 0.607981220657277
[I 2024-04-14 07:03:36,005] Trial 38 finished with value: 0.7271438285276995 and parameters: {'subsample': 0.8218636584157292, 'dropout_rate': 0.11312344832158804, 'n_estimators': 493, 'learning_rate': 0.08480737929792241}. Best is trial 38 with value: 0.7271438285276995.
Fold 1 C-index: 0.7402597402597403
Fold 2 C-index: 0.6383928571428571
Fold 3 C-index: 0.7450980392156863
Fold 4 C-index: 0.7848101265822784
Fold 5 C-index: 0.5985915492957746
[I 2024-04-14 07:03:47,911] Trial 39 finished with value: 0.7014304624992673 and parameters: {'subsample': 0.7648933719827579, 'dropout_rate': 0.3117090493722728, 'n_estimators': 484, 'learning_rate': 0.06141565109530998}. Best is trial 38 with value: 0.7271438285276995.
Fold 1 C-index: 0.7402597402597403
Fold 2 C-index: 0.625
Fold 3 C-index: 0.7352941176470589
Fold 4 C-index

...

Fold 2 IBS: 0.2201721111473965
Fold 3 IBS: 0.16049538705387162
Fold 4 IBS: 0.19883907409387722
Fold 5 IBS: 0.25214311294332836
[I 2024-04-14 07:09:17,718] Trial 19 finished with value: 0.20128574763890333 and parameters: {'subsample': 0.19466785612802578, 'dropout_rate': 0.9970233766526893, 'n_estimators': 400, 'learning_rate': 0.011950521021857646}. Best is trial 16 with value: 0.19758471665097627.
Fold 1 IBS: 0.1735392166010339
Fold 2 IBS: 0.24733020426410265
Fold 3 IBS: 0.17696939073794427
Fold 4 IBS: 0.23474400621151825
Fold 5 IBS: 0.2774794573590098
[I 2024-04-14 07:09:19,221] Trial 20 finished with value: 0.22201245503472178 and parameters: {'subsample': 0.38161734041406226, 'dropout_rate': 0.8176525656681569, 'n_estimators': 260, 'learning_rate': 0.022981610123841693}. Best is trial 16 with value: 0.19758471665097627.
Fold 1 IBS: 0.17538793308166473
Fold 2 IBS: 0.22370044303047862
Fold 3 IBS: 0.16414051314890563
Fold 4 IBS: 0.2099845331024728
Fold 5 IBS: 0.25879976738777416
[I 2

Fold 3 IBS: 0.17002706897662592
Fold 4 IBS: 0.19609017052810865
Fold 5 IBS: 0.2228501539633481
[I 2024-04-14 07:09:40,353] Trial 38 finished with value: 0.19462647659477222 and parameters: {'subsample': 0.1004632969526114, 'dropout_rate': 0.8707974888946499, 'n_estimators': 190, 'learning_rate': 0.013682382045643884}. Best is trial 38 with value: 0.19462647659477222.
Fold 1 IBS: 0.18405040743608886
Fold 2 IBS: 0.1984796262962242
Fold 3 IBS: 0.16110897911275487
Fold 4 IBS: 0.1860026613402482
Fold 5 IBS: 0.23377526281533342
[I 2024-04-14 07:09:41,942] Trial 39 finished with value: 0.19268338740012994 and parameters: {'subsample': 0.10729506851765452, 'dropout_rate': 0.8692930630849747, 'n_estimators': 278, 'learning_rate': 0.013812830432153408}. Best is trial 39 with value: 0.19268338740012994.
Fold 1 IBS: 0.1836380458420524
Fold 2 IBS: 0.22701546500543537
Fold 3 IBS: 0.17208935795899977
Fold 4 IBS: 0.2120304687824635
Fold 5 IBS: 0.24956282584914827
[I 2024-04-14 07:09:43,487] Trial 40 f

Fold 2 IBS: 0.2228601124917642
Fold 3 IBS: 0.16056105379124547
Fold 4 IBS: 0.19023440953991608
Fold 5 IBS: 0.26370761668054876
[I 2024-04-14 07:10:03,506] Trial 57 finished with value: 0.20381402902513912 and parameters: {'subsample': 0.10156433706067643, 'dropout_rate': 0.7395601360396487, 'n_estimators': 270, 'learning_rate': 0.025088218273015078}. Best is trial 39 with value: 0.19268338740012994.
Fold 1 IBS: 0.17402968637702
Fold 2 IBS: 0.22306665859290922
Fold 3 IBS: 0.15697605831771821
Fold 4 IBS: 0.1966965099884976
Fold 5 IBS: 0.25416298400893117
[I 2024-04-14 07:10:04,967] Trial 58 finished with value: 0.20098637945701525 and parameters: {'subsample': 0.1445593583094237, 'dropout_rate': 0.9424651955543935, 'n_estimators': 307, 'learning_rate': 0.017444947774407398}. Best is trial 39 with value: 0.19268338740012994.
Fold 1 IBS: 0.18236249235442123
Fold 2 IBS: 0.22619523389438162
Fold 3 IBS: 0.17099244042885403
Fold 4 IBS: 0.21187950439521488
Fold 5 IBS: 0.24997387585943787
[I 202

Fold 1 IBS: 0.1828122761053612
Fold 2 IBS: 0.2038578816882634
Fold 3 IBS: 0.1690908973158653
Fold 4 IBS: 0.2029501739992507
Fold 5 IBS: 0.24491123804637963
[I 2024-04-14 07:10:13,610] Trial 77 finished with value: 0.20072449343102403 and parameters: {'subsample': 0.23806073249204993, 'dropout_rate': 0.6886627075609139, 'n_estimators': 92, 'learning_rate': 0.04696457264732694}. Best is trial 39 with value: 0.19268338740012994.
Fold 1 IBS: 0.2013436287263864
Fold 2 IBS: 0.19898281712072002
Fold 3 IBS: 0.18358524170353163
Fold 4 IBS: 0.20783372092143054
Fold 5 IBS: 0.21817179832580144
[I 2024-04-14 07:10:13,780] Trial 78 finished with value: 0.20198344135957397 and parameters: {'subsample': 0.14982582293807192, 'dropout_rate': 0.6315405819836285, 'n_estimators': 36, 'learning_rate': 0.03840067689527933}. Best is trial 39 with value: 0.19268338740012994.
Fold 1 IBS: 0.17851295561805824
Fold 2 IBS: 0.21756143717488852
Fold 3 IBS: 0.16876763572866496
Fold 4 IBS: 0.2252965684734787
Fold 5 IBS

Fold 2 IBS: 0.1951009945220422
Fold 3 IBS: 0.17267329733250292
Fold 4 IBS: 0.2002556006238113
Fold 5 IBS: 0.2218911890696791
[I 2024-04-14 07:10:19,223] Trial 96 finished with value: 0.19642633100453882 and parameters: {'subsample': 0.175508689577659, 'dropout_rate': 0.9804710762683706, 'n_estimators': 74, 'learning_rate': 0.029076088824093165}. Best is trial 39 with value: 0.19268338740012994.
Fold 1 IBS: 0.19803585093790257
Fold 2 IBS: 0.1989508054455238
Fold 3 IBS: 0.18076595691737068
Fold 4 IBS: 0.20528537420756332
Fold 5 IBS: 0.2179445399381184
[I 2024-04-14 07:10:19,416] Trial 97 finished with value: 0.20019650548929574 and parameters: {'subsample': 0.15551496658123795, 'dropout_rate': 0.9605705949765594, 'n_estimators': 46, 'learning_rate': 0.033074107860401845}. Best is trial 39 with value: 0.19268338740012994.
Fold 1 IBS: 0.21341308393923883
Fold 2 IBS: 0.2208699224086299
Fold 3 IBS: 0.20400578640567724
Fold 4 IBS: 0.22447238838164785
Fold 5 IBS: 0.21796982004016388
[I 2024-04

In [83]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [84]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.732
train_ibs:  0.193


#### Test

In [85]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [86]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.20442131574078218,
                                              learning_rate=0.08985631890249023,
                                              n_estimators=351,
                                              random_state=123,
                                              subsample=0.10609721872610565)

C-index score: 0.528


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.8692930630849747,
                                              learning_rate=0.013812830432153408,
                                              n_estimators=278,
                                              random_state=123,
                                              subsample=0.10729506851765452)

IBS: 0.266


In [87]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [88]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.826,1.0
ExtraSurvivalTrees,0.794,2.0
CoxPH,0.758,4.0
CoxLasso,0.758,4.0
CoxElastic,0.758,4.0
GradientBoosting,0.745,6.0
ComponentwiseGradientBoosting,0.732,7.0
CoxRidge,0.680,8.0


In [89]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
Randomsurvivalforest,0.183,1.5
ExtraSurvivalTrees,0.183,1.5
CoxLasso,0.187,3.5
CoxElastic,0.187,3.5
CoxPH,0.188,5.0
ComponentwiseGradientBoosting,0.193,6.0
GradientBoosting,0.215,7.0
CoxRidge,0.217,8.0


In [90]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
Randomsurvivalforest,0.653,1.0
ExtraSurvivalTrees,0.618,2.0
GradientBoosting,0.603,3.0
CoxLasso,0.590,4.5
CoxElastic,0.590,4.5
CoxPH,0.585,6.0
CoxRidge,0.534,7.0
ComponentwiseGradientBoosting,0.528,8.0


In [91]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs

,IBS,rank
Randomsurvivalforest,0.206,1.0
ExtraSurvivalTrees,0.215,2.0
GradientBoosting,0.220,3.0
CoxRidge,0.221,4.0
ComponentwiseGradientBoosting,0.266,5.0
CoxLasso,0.284,6.5
CoxElastic,0.284,6.5
CoxPH,0.288,8.0


In [92]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = 'path_to_your_folder/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d1/os/standard/no_selection/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

modified_file_names = ['d1_os_standard_no_selection_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [93]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-14
